In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import jdex.utils.conventions.paths as pc
import json
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator

/Users/navis/.pyenv/versions/kgsaf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
cwd = Path.cwd().parent
dataset_path = cwd / "tutorials/tutorial_dataset"

In [7]:
with open(dataset_path / pc.INDIVIDUAL_MAPPINGS, "r") as f:
    entity_mapping = json.load(f)

with open(dataset_path / pc.OBJ_PROP_MAPPINGS, "r") as f:
    relation_mapping = json.load(f)

In [8]:
train_tf = TriplesFactory.from_path(
    dataset_path / pc.TRAIN,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)

valid_tf = TriplesFactory.from_path(
    dataset_path / pc.VALID,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)

test_tf = TriplesFactory.from_path(
    dataset_path / pc.TEST,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)

In [23]:
from pykeen.pipeline import pipeline

result = pipeline(
    model="TransE",
    training=train_tf,
    validation=valid_tf,
    testing=test_tf,
    model_kwargs=dict(embedding_dim=100),
    training_kwargs=dict(num_epochs=50, batch_size=128),
    device="cpu",
)

INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
/Users/navis/.pyenv/versions/kgsaf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 50/50 [01:41<00:00,  2.03s/epoch, loss=0.00781, prev_loss=0.00669]
Evaluating on cpu:   0%|          | 0.00/2.85k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.85k/2.85k [00:25<00:00, 111triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 25.72s seconds


In [24]:
evaluator = RankBasedEvaluator(filtered=True)

results = evaluator.evaluate(
    model=result.model,
    mapped_triples=test_tf.mapped_triples,
    additional_filter_triples=[
        train_tf.mapped_triples,
        valid_tf.mapped_triples,
    ],
)

Evaluating on cpu:   0%|          | 0.00/2.85k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.85k/2.85k [00:23<00:00, 124triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 23.12s seconds


In [29]:
base = results.to_dict()['both']['realistic']
print(f"Inverse Mean Rank {base['inverse_arithmetic_mean_rank']:5.5f}")
print(f"Mean Rank {base['arithmetic_mean_rank']:5.2f}")
print(f"Hit@10 \t {base['hits_at_10']:4.3f}")
print(f"Hit@5 \t {base['hits_at_5']:4.3f}")
print(f"Hit@1 \t {base['hits_at_1']:4.3f}")

Inverse Mean Rank 0.00023
Mean Rank 4295.88
Hit@10 	 0.159
Hit@5 	 0.129
Hit@1 	 0.020
